# Project 3 (take-home): Risk Estimator App 🩺

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/future-of-mathematicians/notebooks/03_risk_estimator_app.ipynb)

> **From your syllabus**
> - *Applied Probability*: conditional probability, Bayes' rule, expected cost
> - *ML-I*, Unit III: logistic regression

**Your project:** a model that outputs a **probability**, and a decision rule you choose using expected cost. This is where mathematicians earn their salary: the model is easy, the *decision* is the hard part.

> ⚠️ Teaching demo on a public research dataset (Wisconsin Diagnostic Breast Cancer, bundled with scikit-learn). Not medical advice.

⏱ take-home, about 30 minutes · 2 exercises

<details open>
<summary><b>🧭 How this notebook works (30 seconds)</b></summary>

1. **Run a cell:** click it, press **Shift+Enter**. Go **top to bottom, in order**. Collapsed cells titled *"just run me"* must be run too (you do not need to read them).
2. **`____` means your turn.** Replace each `____` with your answer (the **Hint** above the cell helps). Do **not** use *Run all* before the blanks are filled.
3. **Red text?** Read only its **last line**, then check your blank and the hint. Still stuck: ask your squad, then raise a hand. Mistakes are the point.
4. **Core** is the goal for the session. **Stretch** is for when you finish early. **Quest** is take-home. Falling behind? Skip to the next ✔ cell: the maths box matters more than the code.
5. **Colab:** use *File ▸ Save a copy in Drive* to keep your work (the original is read-only).
</details>

> 📊 **Progress tracking: please read.** Run the next cell once: a box will ask for your **name** and **roll number** (press Enter after each). It then draws your progress map and records which exercises you finish. It sends your **name, roll number, exercise ID and time** to the instructor's progress sheet for this workshop, nothing else (no files, nothing from your Google account). Its code is collapsed to save space but is all there to read (Colab: the cell's **⋮** menu → *Show code*; JupyterLab: click the collapsed input). Prefer not to be tracked? Type `anonymous` as your name, or set `TRACKING = False`; the map still works for you. The same cell also turns the error message for an unfilled `____` into a friendly hint.
>
> Colab may ask *"Do you trust this code?"* because this notebook comes from GitHub (`anshid/MathMeetsML`, open source, MIT licence). That is expected: choose **Run anyway**.

In [ ]:
#@title Progress map (just run me)
# Records which exercises you finish so you can see your progress map and the instructor can see how the class is doing.
# Sent: your name, roll number, exercise id and time. Nothing else. Type "anonymous" as your name (or set TRACKING = False) to send nothing.
TRACKING = True
MISSION = 3

import json, os, time, base64, urllib.parse, urllib.request
from IPython.display import HTML, display, update_display

CLOUD_URL = "https://script.google.com/macros/s/AKfycbxekbOkIE5oRVA319mhEENS1phVLt7gPfRUernniP8kMopgIqUNHNN37xjF_3XZEuQWwQ/exec"
TOKEN = "3F5y-xKcPbZZPT49"
LAN_URL = os.environ.get("MML_LAN", "")
MISSIONS = {"1": {"title": "Digit App", "checkpoints": ["p01-ready", "p01-first-model", "p01-model", "p01-app", "p01-done"]}, "2": {"title": "Photo App", "checkpoints": ["p02-svd", "p02-ey", "p02-app", "p02-done"]}, "3": {"title": "Risk App", "checkpoints": ["p03-model", "p03-threshold", "p03-app", "p03-done"]}}
LABEL, UNIT = "P", "Project"
ORDER = sorted(MISSIONS, key=int)
STORE = "progress.json"
SINKS = [u for u in (CLOUD_URL, LAN_URL) if u]


def _load():
    try:
        with open(STORE) as f:
            return json.load(f)
    except Exception:
        return {"name": "", "roll": "", "done": {}, "queue": []}


def _save(s):
    try:
        with open(STORE, "w") as f:
            json.dump(s, f)
    except Exception:
        pass


def _who(s):
    if s["roll"]:
        return
    if os.environ.get("MML_TRACKING") == "off":
        s["name"], s["roll"] = "anonymous", "-"
        return
    try:
        s["name"] = input("Your name (or 'anonymous'): ").strip() or "anonymous"
        s["roll"] = input("Roll number: ").strip() or "-"
    except Exception:
        s["name"], s["roll"] = "anonymous", "-"


def _sending(s):
    return TRACKING and s["name"].lower() != "anonymous" and os.environ.get("MML_TRACKING") != "off"


def _post(url, ev):
    body = json.dumps(dict(ev, token=TOKEN)).encode()
    urllib.request.urlopen(urllib.request.Request(url, body, {"Content-Type": "text/plain"}), timeout=2).read()


def sync():
    """Send queued exercises to the instructor. Silent if offline; they stay queued."""
    s = _load()
    if _sending(s):
        dead = set()
        for ev in s["queue"]:
            for u in SINKS:
                if u in dead or u in ev["sent"]:
                    continue
                try:
                    _post(u, ev)
                    ev["sent"].append(u)
                except Exception:
                    dead.add(u)
        s["queue"] = [e for e in s["queue"] if any(u not in e["sent"] for u in SINKS)]
    _save(s)
    return s


def progress_code():
    """Offline? Paste this code to the instructor (chat or USB) instead."""
    s = _load()
    print(base64.b64encode(json.dumps({"name": s["name"], "roll": s["roll"], "done": s["done"]}).encode()).decode())


def _status(s, n):
    ids = MISSIONS[str(n)]["checkpoints"]
    got = sum(1 for c in ids if c in s["done"])
    if got == len(ids):
        return "done", got, len(ids)
    if n == MISSION or got:
        return "now", got, len(ids)
    return "todo", got, len(ids)


def _html(s):
    cols = {"done": "#2e9e5b", "now": "#3b82f6", "todo": "#8a8f98"}
    parts, stats = [], [_status(s, int(k)) for k in ORDER]
    for i, (st, got, tot) in enumerate(stats):
        n, x = int(ORDER[i]), 45 + i * 108
        title = MISSIONS[str(n)]["title"]
        if i:
            ok = stats[i - 1][0] == "done"
            parts.append(f'<line x1="{x - 108 + 20}" y1="40" x2="{x - 20}" y2="40" stroke="{cols["done"] if ok else cols["todo"]}" stroke-width="3" stroke-dasharray="{"0" if ok else "5 4"}"/>')
        if st == "done":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="{cols[st]}"/><path d="M{x - 8} 40 l6 7 l11 -14" stroke="#fff" stroke-width="3.5" fill="none" stroke-linecap="round" stroke-linejoin="round"/>')
        elif st == "now":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="4"><animate attributeName="r" values="18;22;18" dur="1.8s" repeatCount="indefinite"/></circle><text x="{x}" y="45" text-anchor="middle" font-size="14" font-weight="700" fill="{cols[st]}">{got}/{tot}</text>')
        else:
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="2.5" stroke-dasharray="4 3"/><text x="{x}" y="45" text-anchor="middle" font-size="14" fill="{cols[st]}">{n}</text>')
        parts.append(f'<text x="{x}" y="82" text-anchor="middle" font-size="12" fill="currentColor" opacity="{1 if st != "todo" else 0.55}">{LABEL}{n} {title}</text>')
    finished = [k for k, t in zip(ORDER, stats) if t[0] == "done"]
    todo = [f"{LABEL}{k}" for k, t in zip(ORDER, stats) if t[0] == "todo"]
    now = stats[ORDER.index(str(MISSION))]
    line = f"<b>{len(finished)} of {len(stats)} {UNIT.lower()}s finished</b> &middot; now in <b>{UNIT} {MISSION}</b> ({now[1]}/{now[2]} exercises)"
    line += f" &middot; not visited yet: {', '.join(todo)}" if todo else f" &middot; every {UNIT.lower()} visited"
    who = f"{s['name']} ({s['roll']})" if s["roll"] else ""
    return (f'<div style="font-family:sans-serif;max-width:700px"><svg viewBox="0 0 640 96" width="100%" role="img" aria-label="Progress map">{"".join(parts)}</svg>'
            f'<div style="font-size:14px">{line}</div><div style="font-size:12px;opacity:.6">{who}</div></div>')


_shown = []


def show_tracker():
    html = HTML(_html(_load()))
    if _shown:
        try:
            update_display(html, display_id="mml-tracker")
            return
        except Exception:
            pass
    display(html, display_id="mml-tracker")
    _shown.append(1)


def checkpoint(cid):
    """Call after an exercise passes: saves it, tells the instructor (if online), refreshes the map."""
    s = _load()
    _who(s)
    if cid not in s["done"]:
        s["done"][cid] = time.strftime("%H:%M:%S")
        s["queue"].append({"name": s["name"], "roll": s["roll"], "mission": MISSION, "checkpoint": cid, "ts": s["done"][cid], "sent": []})
    _save(s)
    sync()
    show_tracker()


def _start():
    s = _load()
    _who(s)
    if CLOUD_URL and _sending(s):  # Colab gives every notebook a fresh disk, so ask what you already finished elsewhere
        try:
            q = urllib.parse.urlencode({"roll": s["roll"], "token": TOKEN})
            for c in json.load(urllib.request.urlopen(CLOUD_URL + "?" + q, timeout=2)).get("done", []):
                s["done"].setdefault(c, "earlier")
        except Exception:
            pass
    _save(s)
    show_tracker()


def _friendly_blank(shell, etype, value, tb, tb_offset=None):
    """An unfilled blank (four underscores) raises a cryptic NameError/AttributeError. Say what it means; anything else is shown as usual."""
    blank = "_" * 4
    try:
        if blank in str(value):
            stb = ["✋ You have not filled in a blank yet.",
                   f"   Find the {blank} in this cell, replace it with your answer (the Hint above the cell helps), then run the cell again."]
        else:
            stb = shell.InteractiveTB.structured_traceback(etype, value, tb, tb_offset=tb_offset)
        shell._showtraceback(etype, value, stb)
    except Exception:
        print(f"{etype.__name__}: {value}")


try:
    get_ipython().set_custom_exc((NameError, AttributeError), _friendly_blank)
except Exception:
    pass  # plain Python or an unusual front end: the normal error message is fine


_start()

In [ ]:
#@title ▶ Setup (just run me; code hidden)
import os, sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio==6.31.0"], check=True)
    import gradio as gr

CHECK = os.environ.get("MML_CHECK") == "1"
data = load_breast_cancer()
X, y = data.data, (data.target == 0).astype(int)        # careful: in sklearn 0 = malignant; we recode 1 = malignant
print(X.shape[0], "patients,", X.shape[1], "measurements each; malignant fraction", round(y.mean(), 3))

rng = np.random.default_rng(0)
order = rng.permutation(len(X))
train, test = order[:400], order[400:]

## Stage 1. A model that outputs probabilities
Logistic regression models $P(\text{malignant}\mid x)=\sigma(w^\top x+b)$ with $\sigma(t)=1/(1+e^{-t})$. Fill in the training call (scaling first so every measurement has mean 0, variance 1).

<details><summary>Hint</summary>

The method that learns from data is called `fit`; it takes the inputs and the labels.
</details>

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
model.____(X[train], y[train])                         # TODO: train it
p = model.predict_proba(X[test])[:, 1]                 # P(malignant | x) for each test patient
print("accuracy at threshold 0.5:", round(np.mean((p >= 0.5) == y[test]), 3))
checkpoint("p03-model")

**Are the probabilities honest?** Group patients by predicted probability; in each group, compare the average prediction with the fraction actually malignant (a *calibration* plot). Points near the diagonal mean "70%" really means about 70%.

In [ ]:
bins = np.linspace(0, 1, 6)
which = np.digitize(p, bins[1:-1])
pred_mean = [p[which == b].mean() for b in range(5) if np.any(which == b)]
true_frac = [y[test][which == b].mean() for b in range(5) if np.any(which == b)]
plt.plot([0, 1], [0, 1], "k--", label="perfectly honest"); plt.plot(pred_mean, true_frac, "o-", label="model")
plt.xlabel("predicted P(malignant)"); plt.ylabel("observed fraction malignant"); plt.legend(); plt.show()

## Stage 2. Choose the threshold with expected cost
Refer a patient if $p\ge t$. Missing a cancer (false negative) costs much more than an unnecessary follow-up (false positive). With costs $c_{FN}$ and $c_{FP}$, Bayes decision theory says: refer when
$$c_{FN}\,p > c_{FP}\,(1-p)\iff p>\frac{c_{FP}}{c_{FP}+c_{FN}}.$$
Fill in the expected cost per patient for a threshold $t$, then compare with the formula.

<details><summary>Hint</summary>

Each missed cancer costs `C_FN`, each unnecessary follow-up costs `C_FP`. Multiply each rate (`fn`, `fp`) by its cost and add.
</details>

In [ ]:
C_FN, C_FP = 10.0, 1.0                                 # assumption: a missed cancer is 10x worse than an extra check

def expected_cost(t):
    refer = p >= t
    fn = np.mean(~refer & (y[test] == 1))              # missed cancers, per patient
    fp = np.mean(refer & (y[test] == 0))               # unnecessary follow-ups, per patient
    return ____                                        # TODO: weighted sum of the two error rates

ts = np.linspace(0.01, 0.99, 99)
costs = [expected_cost(t) for t in ts]
best = ts[int(np.argmin(costs))]
plt.plot(ts, costs); plt.axvline(C_FP / (C_FP + C_FN), color="r", ls="--", label="Bayes rule t* = c_FP/(c_FP+c_FN)")
plt.xlabel("threshold t"); plt.ylabel("expected cost per patient"); plt.legend(); plt.show()
print(f"best threshold on test data {best:.2f}, Bayes formula {C_FP / (C_FP + C_FN):.2f}")
assert expected_cost(0.5) > 0 and min(costs) <= expected_cost(0.5)
checkpoint("p03-threshold")

> **Think like a mathematician.** The best empirical threshold on 169 test patients is noisy; the Bayes formula is principled *if* the probabilities are calibrated. Which would you defend to a hospital? The app uses the Bayes value rounded, $t=0.10$: far below 0.5, because missing a cancer costs more. If your cost ratio differs, change `THRESHOLD` in `APP_PY`.

## Stage 3. The app
Five sliders for the most informative measurements; everything else fixed at the median.

In [ ]:
APP_PY = r'''# app.py - Tumour Risk Estimator, a teaching demo (MathMeetsML project). NOT medical advice.
import numpy as np
import gradio as gr
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer()
X, y = data.data, (data.target == 0).astype(int)        # 1 = malignant, 0 = benign
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(X, y)
FEATURES = ["worst radius", "worst concave points", "worst texture", "mean smoothness", "worst symmetry"]
IDX = [list(data.feature_names).index(f) for f in FEATURES]
TYPICAL = np.median(X, axis=0)
THRESHOLD = 0.10                                         # Bayes rule c_FP/(c_FP+c_FN) with a 10:1 cost ratio, rounded


def risk(*values):
    x = TYPICAL.copy()
    x[IDX] = values
    p = float(model.predict_proba([x])[0, 1])
    flag = "refer for follow-up" if p >= THRESHOLD else "below the referral threshold"
    return {"malignant": p, "benign": 1 - p}, f"P(malignant | measurements) = {p:.1%} -> {flag} (threshold {THRESHOLD:.0%})"


sliders = [gr.Slider(float(X[:, i].min()), float(X[:, i].max()), value=float(TYPICAL[i]), label=f) for f, i in zip(FEATURES, IDX)]
demo = gr.Interface(fn=risk, inputs=sliders, outputs=[gr.Label(label="Model probability"), gr.Textbox(label="Decision")],
                    title="Tumour Risk Estimator (teaching demo, not medical advice)",
                    description="Logistic regression on the Wisconsin breast-cancer dataset. Other measurements are fixed at their median.",
                    flagging_mode="never")

if __name__ == "__main__":
    demo.launch()'''
exec(APP_PY.replace('if __name__ == "__main__":', "if False:"))

probs, decision = risk(*[float(np.median(X[:, i])) for i in IDX])
print(decision)
if CHECK:
    print("check mode: app built, not launched")
else:
    demo.launch(share=True)
checkpoint("p03-app")

## Stage 4. Package it
Edit `YOUR_NAME` in the first cell, then run both cells.

In [ ]:
YOUR_NAME = "Your Name"                            # TODO (optional): your name for the README

In [ ]:
#@title ▶ Write, test and zip the project (just run me; code hidden)
import runpy, shutil
README = f'''---
title: Tumour Risk Estimator (teaching demo)
emoji: 🩺
sdk: gradio
sdk_version: {gr.__version__}
app_file: app.py
license: mit
---
# Tumour Risk Estimator (teaching demo, not medical advice)

Logistic regression gives $P(\\text{{malignant}}\\mid x)=\\sigma(w^\\top x+b)$ on the Wisconsin Diagnostic Breast Cancer data.
The referral threshold comes from Bayes decision theory, $t^*=c_{{FP}}/(c_{{FP}}+c_{{FN}})$, with a 10:1 cost ratio,
compared against the empirical expected-cost curve. Calibration is checked with a reliability plot.

Built by {YOUR_NAME} at the MathMeetsML workshop. Code MIT. Dataset: UCI / scikit-learn (W. Wolberg, W. Street, O. Mangasarian).
'''
os.makedirs("risk-estimator", exist_ok=True)
for fname, text in {"app.py": APP_PY, "README.md": README, "requirements.txt": "scikit-learn==1.9.1\nnumpy\n"}.items():
    with open(os.path.join("risk-estimator", fname), "w", encoding="utf-8") as f:
        f.write(text.strip() + "\n")
packed = runpy.run_path("risk-estimator/app.py", run_name="packaged")
assert 0 <= packed["risk"](*packed["TYPICAL"][packed["IDX"]])[0]["malignant"] <= 1
shutil.make_archive("risk-estimator", "zip", "risk-estimator")
if not CHECK:
    try:
        from google.colab import files
        files.download("risk-estimator.zip")
    except ImportError:
        print("Not on Colab: the zip is next to this notebook.")
print("packaged and tested ✔  ->  risk-estimator.zip")
checkpoint("p03-done")

## Stretch
- Change the cost ratio to 50:1. How many extra follow-ups buy how many fewer missed cancers?
- Replace the five sliders with the five features with the largest $|w_j|$ (after scaling). Are they the same?

## You just used...
- **conditional probability** and **Bayes decision theory** (Applied Probability)
- **logistic regression** as a probability model, and **calibration**
- the habit that separates analysts from button-pressers: making assumptions (the cost ratio) explicit

## Your CV line
> *Built and deployed a probabilistic risk-estimation web app (scikit-learn, Gradio) with a calibration check and a Bayes-optimal decision threshold derived from explicit error costs.*